# TORGO Sentences EDA

Exploratory data analysis of the TORGO dataset (sentence-level utterances).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
%matplotlib inline

## Load Manifest

In [ ]:
manifest = pd.read_csv('../manifests/torgo_sentences.csv')
print(f"Total utterances: {len(manifest)}")
manifest.head()

## Speaker Statistics

In [ ]:
speaker_counts = manifest['speaker_id'].value_counts()
print(f"Number of speakers: {len(speaker_counts)}")
print("\nUtterances per speaker:")
print(speaker_counts)

In [ ]:
# Plot distribution
plt.figure(figsize=(12, 5))
speaker_counts.plot(kind='bar')
plt.title('Number of Utterances per Speaker')
plt.xlabel('Speaker ID')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## Duration Analysis

In [ ]:
# Duration stats (if available in manifest)
if 'duration' in manifest.columns and manifest['duration'].notna().any():
    print("Duration statistics:")
    print(manifest['duration'].describe())
    
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    manifest['duration'].hist(bins=30)
    plt.title('Duration Distribution')
    plt.xlabel('Duration (seconds)')
    plt.ylabel('Count')
    
    plt.subplot(1, 2, 2)
    manifest.boxplot(column='duration', by='speaker_id', ax=plt.gca())
    plt.title('Duration by Speaker')
    plt.suptitle('')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Duration not available in manifest. Run with audio files to compute.")

## Missing Files Check

In [ ]:
from pathlib import Path

missing = []
for _, row in manifest.iterrows():
    if not Path(row['path']).exists():
        missing.append(row['path'])

print(f"Missing files: {len(missing)}")
if missing:
    print("Examples:", missing[:5])

## Session Distribution

In [ ]:
session_counts = manifest.groupby(['speaker_id', 'session']).size().unstack(fill_value=0)
print(session_counts)

# Heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(session_counts, annot=True, fmt='d', cmap='YlOrRd')
plt.title('Utterances by Speaker and Session')
plt.tight_layout()
plt.show()

## Summary for Week 1 Pilot

Select 2-3 speakers for pilot subset.

In [ ]:
# Select pilot speakers (those with most data, for example)
pilot_speakers = speaker_counts.head(3).index.tolist()
print(f"Pilot speakers: {pilot_speakers}")

pilot_manifest = manifest[manifest['speaker_id'].isin(pilot_speakers)]
print(f"\nPilot subset size: {len(pilot_manifest)} utterances")

# Save pilot manifest
pilot_manifest.to_csv('../manifests/torgo_pilot.csv', index=False)
print("Saved to manifests/torgo_pilot.csv")